# Multiple Related Series

Real-world time series rarely live in isolation. Exchange rates move together,
commodity prices transmit shocks across markets, and macroeconomic indicators
feed back on one another. This notebook explores **three pillars** of
multivariate time-series analysis:

| Pillar | Question answered | Key tool |
|--------|-------------------|----------|
| **Joint dynamics** | How do series co-evolve over time? | `var_fit` / `var_forecast` |
| **Causality** | Does one series help predict another? | `granger_causality` |
| **Volatility clustering** | Does uncertainty itself cluster? | `garch_fit` / `garch_forecast` |

We work entirely with synthetic data so every ground-truth parameter is known.

**References**
- Lütkepohl, H. (2005). *New Introduction to Multiple Time Series Analysis*. Springer.
- Bollerslev, T. (1986). Generalized Autoregressive Conditional Heteroskedasticity. *Journal of Econometrics*, 31(3), 307–327.
- Granger, C. W. J. (1969). Investigating Causal Relations by Econometric Models and Cross-spectral Methods. *Econometrica*, 37(3), 424–438.

## 1 &mdash; Imports and synthetic data

In [ ]:
try:
    import hvplot.polars  # noqa
except ImportError:
    pass  # hvplot is optional for visualization
import numpy as np
import polars as pl

from polars_ts import (
    acf,
    garch_fit,
    garch_forecast,
    granger_causality,
    ljung_box,
    pacf,
    var_fit,
    var_forecast,
)

In [ ]:
np.random.seed(42)
n = 300

# ---------- VAR-like correlated pair ----------
y1, y2 = [0.0], [0.0]
for _t in range(1, n):
    y1.append(0.5 * y1[-1] + 0.3 * y2[-1] + np.random.normal(0, 1))
    y2.append(0.2 * y1[-2] + 0.4 * y2[-1] + np.random.normal(0, 1.5))

# ---------- GARCH returns ----------
returns = []
sigma = 1.0
for _t in range(n):
    returns.append(np.random.normal(0, sigma))
    sigma = np.sqrt(0.1 + 0.3 * returns[-1] ** 2 + 0.5 * sigma**2)

# ---------- Polars DataFrames ----------
df_var = pl.DataFrame(
    {
        "ds": list(range(n)),
        "y1": y1,
        "y2": y2,
    }
)

df_garch = pl.DataFrame(
    {
        "unique_id": ["R1"] * n,
        "ds": list(range(n)),
        "y": returns,
    }
)

print(f"VAR data: {df_var.shape}")
print(f"GARCH data: {df_garch.shape}")
df_var.head()

## 2 &mdash; Exploratory analysis

Before fitting any model we visualize the raw series and inspect their
autocorrelation structure with `acf` and `pacf`.

In [ ]:
# Plot the two correlated series side by side
df_var.hvplot.line(
    x="ds",
    y=["y1", "y2"],
    title="Correlated pair (VAR DGP)",
    width=800,
    height=350,
    ylabel="value",
)

In [ ]:
# Reshape for grouped ACF / PACF
df_long = pl.concat(
    [
        df_var.select(
            pl.lit(col).alias("unique_id"),
            pl.col("ds"),
            pl.col(col).alias("y"),
        )
        for col in ["y1", "y2"]
    ]
)

acf_df = acf(df_long, max_lags=20)
pacf_df = pacf(df_long, max_lags=20)

acf_plot = acf_df.hvplot.bar(
    x="lag",
    y="acf",
    by="unique_id",
    title="ACF by series",
    width=800,
    height=300,
    rot=0,
)
pacf_plot = pacf_df.hvplot.bar(
    x="lag",
    y="pacf",
    by="unique_id",
    title="PACF by series",
    width=800,
    height=300,
    rot=0,
)
acf_plot + pacf_plot

## 3 &mdash; Ljung-Box test

The Ljung-Box Q statistic tests the null hypothesis that the first *k*
autocorrelations are jointly zero. A small p-value tells us that significant
autocorrelation remains and modelling is worthwhile.

In [ ]:
lb_results = ljung_box(df_long, lags=[10, 20])
lb_results

Both series should reject the null at conventional significance levels,
confirming the autocorrelation we saw in the ACF plots.

## 4 &mdash; VAR model

A **Vector Autoregression** of order *p* models every series as a linear
function of the last *p* values of *all* series in the system.

Our data-generating process uses $p = 1$ lags of each series, so fitting
with $p = 2$ gives the model room to discover that the second lag is
approximately zero.

In [ ]:
var_model = var_fit(df_var, target_cols=["y1", "y2"], p=2)

print("Coefficient matrix (each row = one equation):")
print(var_model.coefficients)
print(f"\nResidual shape: {var_model.residuals.shape}")

In [ ]:
# Produce a 20-step-ahead forecast
fc = var_forecast(var_model, horizon=20)

# Overlay forecast on the tail of the actuals
tail = df_var.tail(60).with_columns(pl.lit("actual").alias("source"))
fc_plot = fc.with_columns(pl.lit("forecast").alias("source"))

combined = pl.concat([tail, fc_plot], how="diagonal_relaxed")

combined.hvplot.line(
    x="ds",
    y="y1",
    by="source",
    title="VAR forecast – y1",
    width=800,
    height=350,
) + combined.hvplot.line(
    x="ds",
    y="y2",
    by="source",
    title="VAR forecast – y2",
    width=800,
    height=350,
)

## 5 &mdash; Granger causality

Granger causality asks: *does past y1 help predict y2, over and above y2's
own past?* By construction, y1 feeds into y2 (coefficient 0.2) and y2 feeds
into y1 (coefficient 0.3), so we expect **bi-directional** causality.

In [ ]:
gc = granger_causality(df_var, target_cols=["y1", "y2"], lag=5)
gc

## 6 &mdash; GARCH for volatility clustering

Financial returns often exhibit *volatility clustering*: large moves beget
large moves. A **GARCH(1,1)** model captures this through:

$$\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2$$

Our synthetic returns were generated with $\omega = 0.1$, $\alpha = 0.3$,
$\beta = 0.5$.

In [ ]:
garch_models = garch_fit(df_garch, p=1, q=1)

for uid, m in garch_models.items():
    print(f"--- {uid} ---")
    print(f"  omega = {m.omega:.4f}  (true 0.1)")
    print(f"  alpha = {m.alpha:.4f}  (true 0.3)")
    print(f"  beta  = {m.beta:.4f}  (true 0.5)")

In [ ]:
# Visualize in-sample conditional variance
cond_var = garch_models["R1"].conditional_variance

df_vol = pl.DataFrame(
    {
        "ds": list(range(len(cond_var))),
        "conditional_variance": cond_var,
        "squared_return": [r**2 for r in returns[: len(cond_var)]],
    }
)

df_vol.hvplot.line(
    x="ds",
    y=["conditional_variance", "squared_return"],
    title="GARCH(1,1) conditional variance vs squared returns",
    width=800,
    height=350,
    ylabel="variance",
    alpha=0.7,
)

## 7 &mdash; GARCH forecast

In [ ]:
garch_fc = garch_forecast(garch_models["R1"], horizon=20)

df_garch_fc = pl.DataFrame(
    {
        "ds": list(range(n, n + len(garch_fc))),
        "forecast_variance": garch_fc,
    }
)

# Append forecast to in-sample conditional variance
in_sample = df_vol.select("ds", pl.col("conditional_variance").alias("variance")).with_columns(
    pl.lit("in-sample").alias("source")
)
out_sample = df_garch_fc.select("ds", pl.col("forecast_variance").alias("variance")).with_columns(
    pl.lit("forecast").alias("source")
)

pl.concat([in_sample.tail(80), out_sample]).hvplot.line(
    x="ds",
    y="variance",
    by="source",
    title="GARCH variance forecast (20-step ahead)",
    width=800,
    height=350,
)

## 8 &mdash; Summary

| Step | What we learned |
|------|----------------|
| **ACF / PACF** | Both series show significant autocorrelation, justifying a VAR model. |
| **Ljung-Box** | Formal confirmation that autocorrelation is statistically significant. |
| **VAR(2)** | Captures the cross-series dynamics; forecast reverts to the mean. |
| **Granger causality** | Bi-directional causality detected, matching the true DGP. |
| **GARCH(1,1)** | Recovers volatility-clustering parameters close to ground truth. |
| **GARCH forecast** | Variance forecast converges to the unconditional variance $\omega / (1 - \alpha - \beta)$. |

### Next steps

- Extend to **multivariate GARCH** (DCC, BEKK) once the API supports it.
- Combine VAR residuals with GARCH to build a **VAR-GARCH** model.
- Apply to real financial data (e.g., equity index returns from Yahoo Finance).